<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/ml-labs/RoBERTaModelTrainedWeek10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Example using RoBERTa model trained on Twitter data

In [23]:
# sentiment_llm_pyspark.py

from pyspark.sql import SparkSession
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score
import warnings

# Ignore warnings from Hugging Face
warnings.filterwarnings("ignore")

# 1. Create sample data
data = {
    "sentence": [
        "I love this product!", "This is the worst service ever.", "Absolutely fantastic experience.",
        "I'm not happy with the results.", "The movie was okay, not great.", "What a wonderful surprise!",
        "I would not recommend this.", "Such a delightful day.", "Terrible customer support.",
        "This phone is amazing!", "Very disappointing performance.", "I'm so excited about this!",
        "Could be better.", "Totally satisfied with my purchase.", "I hate how this works.",
        "It exceeded my expectations!", "Nothing special about it.", "I'm impressed by the quality.",
        "Worst purchase I've made.", "A pretty decent option."
    ],
    "label": [
        "positive", "negative", "positive", "negative", "neutral", "positive", "negative",
        "positive", "negative", "positive", "negative", "positive", "neutral", "positive",
        "negative", "positive", "neutral", "positive", "negative", "neutral"
    ]
}

In [24]:
# 2. Start Spark session
spark = SparkSession.builder.appName("LLM Sentiment Evaluation").getOrCreate()

In [25]:
# 3. Convert data to Spark DataFrame
df_pd = pd.DataFrame(data)
df_spark = spark.createDataFrame(df_pd)

In [26]:
# 4. Convert Spark → Pandas for inference
df = df_spark.toPandas()

In [27]:
# 5. Load Hugging Face sentiment analysis model
classifier = pipeline("sentiment-analysis")  # Defaults to distilbert-base-uncased-finetuned-sst-2-english

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [28]:
# 6. Run predictions
def map_prediction(pred):
    label = pred['label'].lower()
    if label == 'positive':
        return 'positive'
    elif label == 'negative':
        return 'negative'
    else:
        return 'neutral'

df['predicted'] = df['sentence'].apply(lambda x: map_prediction(classifier(x)[0]))

In [29]:
# 7. Evaluate results
print("\nClassification Report:")
print(classification_report(df['label'], df['predicted'], digits=3))

print("\nAccuracy Score:", accuracy_score(df['label'], df['predicted']))


Classification Report:
              precision    recall  f1-score   support

    negative      0.700     1.000     0.824         7
     neutral      0.000     0.000     0.000         4
    positive      0.900     1.000     0.947         9

    accuracy                          0.800        20
   macro avg      0.533     0.667     0.590        20
weighted avg      0.650     0.800     0.715        20


Accuracy Score: 0.8


In [30]:
# 8. Convert back to Spark for further processing if needed
df_result_spark = spark.createDataFrame(df)
df_result_spark.show(truncate=False)

+-----------------------------------+--------+---------+
|sentence                           |label   |predicted|
+-----------------------------------+--------+---------+
|I love this product!               |positive|positive |
|This is the worst service ever.    |negative|negative |
|Absolutely fantastic experience.   |positive|positive |
|I'm not happy with the results.    |negative|negative |
|The movie was okay, not great.     |neutral |negative |
|What a wonderful surprise!         |positive|positive |
|I would not recommend this.        |negative|negative |
|Such a delightful day.             |positive|positive |
|Terrible customer support.         |negative|negative |
|This phone is amazing!             |positive|positive |
|Very disappointing performance.    |negative|negative |
|I'm so excited about this!         |positive|positive |
|Could be better.                   |neutral |negative |
|Totally satisfied with my purchase.|positive|positive |
|I hate how this works.        

In [31]:
# 9. Stop Spark
spark.stop()

#### To apply a Hugging Face model to Spark DataFrame without using pandas, the best approach is: Use a PySpark UDF (User Defined Function)

In [32]:
# pyspark_hf_sentiment_udf.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from transformers import pipeline

# Step 1: Start Spark session
spark = SparkSession.builder.appName("SparkLLMSentiment").getOrCreate()

# Step 2: Sample dataset (20 sentences with various sentiments)
data = [
    ("I love this product!",),
    ("This is the worst service ever.",),
    ("Absolutely fantastic experience.",),
    ("I'm not happy with the results.",),
    ("The movie was okay, not great.",),
    ("What a wonderful surprise!",),
    ("I would not recommend this.",),
    ("Such a delightful day.",),
    ("Terrible customer support.",),
    ("This phone is amazing!",),
    ("Very disappointing performance.",),
    ("I'm so excited about this!",),
    ("Could be better.",),
    ("Totally satisfied with my purchase.",),
    ("I hate how this works.",),
    ("It exceeded my expectations!",),
    ("Nothing special about it.",),
    ("I'm impressed by the quality.",),
    ("Worst purchase I've made.",),
    ("A pretty decent option.",)
]

columns = ["sentence"]
df = spark.createDataFrame(data, columns)

In [33]:
# Step 3: Define UDF that wraps Hugging Face model
def load_model():
    return pipeline("sentiment-analysis")

def predict_sentiment(text):
    global clf
    if "clf" not in globals():
        clf = load_model()
    result = clf(text)[0]['label'].lower()
    return result

In [34]:
# Step 4: Register as UDF
sentiment_udf = udf(predict_sentiment, StringType())

In [35]:
# Step 5: Apply UDF to Spark DataFrame
df_with_predictions = df.withColumn("predicted_sentiment", sentiment_udf(df["sentence"]))

In [36]:
# Step 6: Show results
df_with_predictions.show(truncate=False)

# Optional: Save to CSV
# df_with_predictions.write.csv("sentiment_output.csv", header=True, mode="overwrite")

+-----------------------------------+-------------------+
|sentence                           |predicted_sentiment|
+-----------------------------------+-------------------+
|I love this product!               |positive           |
|This is the worst service ever.    |negative           |
|Absolutely fantastic experience.   |positive           |
|I'm not happy with the results.    |negative           |
|The movie was okay, not great.     |negative           |
|What a wonderful surprise!         |positive           |
|I would not recommend this.        |negative           |
|Such a delightful day.             |positive           |
|Terrible customer support.         |negative           |
|This phone is amazing!             |positive           |
|Very disappointing performance.    |negative           |
|I'm so excited about this!         |positive           |
|Could be better.                   |negative           |
|Totally satisfied with my purchase.|positive           |
|I hate how th

In [37]:
# Step 7: Stop Spark session
spark.stop()